# Python Advanced - Day 2 Homework

**Runbook alignment (Day 2, Hours 5-8):**
- Hour 5: Pattern: Factory (practical creation + validation)
- Hour 6: Pattern: Strategy (swap behaviors cleanly)
- Hour 7: Pythonic class ergonomics (repr/str/equality) + light type hints
- Hour 8: Checkpoint 1: Domain + service layer milestone

This homework continues the full-stack tracker domain by centralizing object creation, swapping behaviors, polishing class ergonomics, and proving the service layer checkpoint.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day2.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 5 - Pattern: Factory (practical creation + validation)

**Goal:** Build a validated factory that turns a raw payload into a safe tracker record.

**Best practice:** Use a classmethod factory to validate and normalize payloads before object creation.

**Avoid this pitfall:** Duplicating validation logic in every caller leads to inconsistent objects.


In [ ]:
# Hour 5 scaffold: Factory pattern with validation + normalization
from dataclasses import dataclass, field


@dataclass
class TaskRecord:
    title: str
    priority: str
    owner: str = "unassigned"
    tags: list[str] = field(default_factory=list)

    @classmethod
    def from_payload(cls, payload: dict[str, object]) -> "TaskRecord":
        title = str(payload.get("title", "")).strip()
        if not title:
            raise ValueError("title must not be blank")

        priority = str(payload.get("priority", "medium")).strip().lower() or "medium"
        owner = str(payload.get("owner", "unassigned")).strip() or "unassigned"
        raw_tags = payload.get("tags", [])
        tags = [str(tag).strip().lower() for tag in raw_tags if str(tag).strip()]
        return cls(title=title, priority=priority, owner=owner, tags=tags)


factory_task = TaskRecord.from_payload(
    {
        "title": "Refactor auth flow",
        "priority": "HIGH",
        "tags": ["API", "backend"],
    }
)

try:
    TaskRecord.from_payload({"title": "   ", "priority": "low"})
except ValueError as exc:
    factory_error = str(exc)

print(f"Hour 5 | factory title: {factory_task.title}")
print(f"Hour 5 | factory priority: {factory_task.priority}")
print(f"Hour 5 | factory owner: {factory_task.owner}")
print(f"Hour 5 | factory tags: {', '.join(factory_task.tags)}")
print(f"Hour 5 | validation error: {factory_error}")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 6 - Pattern: Strategy (swap behaviors cleanly)

**Goal:** Build a service that can switch notification or sorting behavior without rewriting the caller.

**Best practice:** Inject behavior objects or callables instead of hard-coding branching logic.

**Avoid this pitfall:** Burying every algorithm inside one if or elif chain makes behavior swaps harder.


In [ ]:
# Hour 6 scaffold: Strategy pattern with swappable notification behaviors
from typing import Protocol


class Notifier(Protocol):
    def send(self, text: str) -> str:
        ...


class EmailNotifier:
    def send(self, text: str) -> str:
        return f"emailed {text}"


class SlackNotifier:
    def send(self, text: str) -> str:
        return f"posted {text} to slack"


class TrackerAlertService:
    def __init__(self, notifier: Notifier) -> None:
        self.notifier = notifier

    def set_notifier(self, notifier: Notifier) -> None:
        self.notifier = notifier

    def announce_update(self) -> str:
        return self.notifier.send("tracker update")


service = TrackerAlertService(EmailNotifier())
first_send = service.announce_update()
service.set_notifier(SlackNotifier())
second_send = service.announce_update()

print("Hour 6 | strategy selected: EmailNotifier")
print(f"Hour 6 | first send: {first_send}")
print("Hour 6 | swapped strategy: SlackNotifier")
print(f"Hour 6 | second send: {second_send}")
print("Hour 6 | strategy rule: same service, different behavior")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 3: Hour 7 - Pythonic class ergonomics (repr/str/equality) + light type hints

**Goal:** Build a class whose string output and equality semantics support debugging and testing.

**Best practice:** Implement repr, str, and equality so debugging and comparisons are clear.

**Avoid this pitfall:** Leaving model output unreadable makes logs and tests harder to interpret.


In [ ]:
# Hour 7 scaffold: class ergonomics for debugging + testing
from dataclasses import dataclass


@dataclass(eq=True)
class Task:
    title: str
    priority: str
    done: bool = False

    def __repr__(self) -> str:
        return f"Task(title='{self.title}', priority='{self.priority}')"

    def __str__(self) -> str:
        return f"{self.title} [{self.priority}]"


task_a = Task(title="Write tests", priority="medium")
task_b = Task(title="Write tests", priority="medium")
task_c = Task(title="Write tests", priority="high")

print(f"Hour 7 | repr: {repr(task_a)}")
print(f"Hour 7 | str: {task_a}")
print(f"Hour 7 | equality same fields: {task_a == task_b}")
print(f"Hour 7 | equality different priority: {task_a == task_c}")
print("Hour 7 | type hints: title=str, priority=str, done=bool")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 4: Hour 8 - Checkpoint 1: Domain + service layer milestone

**Goal:** Build a small service layer milestone that adds, validates, and lists tracker records.

**Best practice:** Keep domain logic in the service layer and return structured results for the checkpoint demo.

**Avoid this pitfall:** Letting the UI or raw input dictionaries bypass the service layer weakens the design.


In [ ]:
# Hour 8 scaffold: service layer checkpoint with validation boundary
from dataclasses import dataclass


@dataclass
class TrackerTask:
    title: str
    priority: str


class TrackerService:
    def __init__(self) -> None:
        self._records: list[TrackerTask] = []

    def add_task(self, payload: dict[str, str]) -> None:
        title = payload.get("title", "").strip()
        if not title:
            raise ValueError("title must not be blank")
        priority = payload.get("priority", "medium").strip().lower() or "medium"
        self._records.append(TrackerTask(title=title, priority=priority))

    def list_tasks(self) -> list[TrackerTask]:
        return list(self._records)


checkpoint_service = TrackerService()
checkpoint_service.add_task({"title": "Plan sprint", "priority": "high"})
checkpoint_service.add_task({"title": "Write changelog", "priority": "medium"})

invalid_add_result = "accepted"
try:
    checkpoint_service.add_task({"title": "   ", "priority": "low"})
except ValueError:
    invalid_add_result = "rejected"

records = checkpoint_service.list_tasks()
priority_summary = ", ".join(task.priority for task in records)

print(f"Hour 8 | checkpoint records: {len(records)}")
print(f"Hour 8 | checkpoint first task: {records[0].title}")
print(f"Hour 8 | checkpoint priorities: {priority_summary}")
print(f"Hour 8 | checkpoint invalid add: {invalid_add_result}")
print("Hour 8 | checkpoint status: domain and service layer ready")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day2_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


In [ ]:
# Integrity note:
# This notebook is intentionally scaffolded for authentic implementation work.
# Avoid replacing the assignment with a hard-coded print-only script.
# Use your own classes/functions above and keep outputs deterministic for grading.
